In [ ]:
import os
import pickle
import re
from pathlib import Path
import pandas as pd
import numpy as np
from utils.model_saver import *

PROJECT_ROOT = Path().resolve().parent.parent
print(f"Project root: {PROJECT_ROOT}")

PATH_DATA = PROJECT_ROOT / 'data'
PATH_NORMAL_DATA = PATH_DATA / 'normal_features'
PATH_GRAPH_DATA = PATH_DATA / 'graph_features'
PATH_TEXT_DATA = PATH_DATA / 'textual_features'
PATH_COMBINED_DATA = PATH_DATA / 'combined_features'
MODELS_SAVE_PATH = PROJECT_ROOT / 'Models'

SAVE = True
PATH_PLOTS = PROJECT_ROOT / 'report' / 'src'
TARGET = 'is_reference_valid'
RANDOM_STATE = 42

import torch
# Detect device: 'cuda' if available, else 'cpu'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {DEVICE}")

# for parallelization
N_JOBS = -1

Using device: cpu
Project root: C:\Users\ivan.bernacchia\OneDrive - TINEXT SA\Documents\_Alessia\M-P6203E-DataProjects-Hackaton3_P1
Using device: cpu


In [ ]:
ID_COLUMNS = ["article_id", "ref_id"]

norm_train = pd.read_parquet(PATH_NORMAL_DATA / 'train.parquet')
norm_test = pd.read_parquet(PATH_NORMAL_DATA / 'test.parquet')
norm_train = norm_train.drop(columns=ID_COLUMNS, errors="ignore")
norm_test = norm_test.drop(columns=ID_COLUMNS, errors="ignore")

graph_train = pd.read_parquet(PATH_GRAPH_DATA / 'final_train.parquet')
graph_test = pd.read_parquet(PATH_GRAPH_DATA / 'final_test.parquet')
graph_train = graph_train.drop(columns=ID_COLUMNS, errors="ignore")
graph_test = graph_test.drop(columns=ID_COLUMNS, errors="ignore")

df_64 = pd.read_parquet(PATH_TEXT_DATA / 'textual_embeddings_64.parquet')
# splitting
split_series = df_64["split"].astype(str).str.lower()
text64_train = df_64[split_series == "train"].copy()
text64_test = df_64[split_series == "test"].copy()
text64_train = text64_train.drop(columns=ID_COLUMNS, errors="ignore")
text64_test = text64_test.drop(columns=ID_COLUMNS, errors="ignore")

df_128 = pd.read_parquet(PATH_TEXT_DATA / 'textual_embeddings_128.parquet')
# splitting
split_series = df_128["split"].astype(str).str.lower()
text128_train = df_128[split_series == "train"].copy()
text128_test = df_128[split_series == "test"].copy()
text128_train = text128_train.drop(columns=ID_COLUMNS, errors="ignore")
text128_test = text128_test.drop(columns=ID_COLUMNS, errors="ignore")

mix_train = pd.read_parquet(PATH_COMBINED_DATA / 'train.parquet')
mix_test = pd.read_parquet(PATH_COMBINED_DATA / 'test.parquet')
mix_train = mix_train.drop(columns=ID_COLUMNS, errors="ignore")
mix_test = mix_test.drop(columns=ID_COLUMNS, errors="ignore")

In [4]:
import os
import pickle
from pathlib import Path

def load_latest_models(base_path='./Models'):
    """
    Creates a nested dictionary registry.
    Access via: registry['folder_name']['KNN']
    """
    models_registry = {}
    base_dir = Path(base_path)

    if not base_dir.exists():
        print(f"Error: Directory {base_path} not exist.")
        return models_registry

    # Define the keywords to identify the three model types
    model_types = {
        'KNN': 'Best_KNN',
        'XGB': 'Best_XGB',
        'transformer': 'Transformer'
    }

    for subdir in base_dir.iterdir():
        if subdir.is_dir():
            models_registry[subdir.name] = {}
            
            # For each folder, look for the 3 specific model types
            for label, keyword in model_types.items():
                # Filter files that contain the keyword and end in .pkl
                model_files = [f for f in subdir.glob(f'*{keyword}*.pkl')]
                
                if not model_files:
                    continue

                # Sort alphabetically to get the latest timestamp at the end
                latest_file = sorted(model_files, key=lambda x: x.name)[-1]
                
                try:
                    with open(latest_file, 'rb') as f:
                        models_registry[subdir.name][label] = pickle.load(f)
                    print(f"Loaded {label} from {subdir.name}: {latest_file.name}")
                except Exception as e:
                    print(f"Failed to load {label} in {subdir.name}: {e}")

    return models_registry

import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix, f1_score, accuracy_score, precision_score, recall_score

def evaluate_model_on_sets(model, set_dict):
    """
    Evaluates a single model on its specific dictionary of sets.
    Returns a dict with confusion matrices and a dataframe of metrics.
    """
    results = {'cms': {}, 'metrics': []}
    
    for set_name, (X, y) in set_dict.items():
        # Predict using the standardized interface
        y_pred = model.predict(X)
        
        # Calculate Confusion Matrix
        results['cms'][set_name] = confusion_matrix(y, y_pred)
        
        # Calculate Weighted Metrics
        results['metrics'].append({
            'Set': set_name,
            'F1': f1_score(y, y_pred, average='weighted'),
            'Accuracy': accuracy_score(y, y_pred),
            'Precision': precision_score(y, y_pred, average='weighted'),
            'Recall': recall_score(y, y_pred, average='weighted')
        })
    
    results['metrics_df'] = pd.DataFrame(results['metrics']).set_index('Set')
    return results

import matplotlib.pyplot as plt
import seaborn as sns

def plot_model_comparison(model_list, model_names, all_sets_list, title="Model Comparison", figsize=(18, 10)):
    """
    Args:
        model_list: List of trained models.
        model_names: List of strings (e.g., ['KNN', 'XGB', 'Transformer']).
        all_sets_list: List of dicts, each dict contains {'train': (X, y), 'test': (X, y)}.
        title: Main title of the figure.
    """
    n_models = len(model_list)
    set_names = list(all_sets_list[0].keys()) # Typically ['train', 'test']
    n_sets = len(set_names)
    
    # We add one extra column for the horizontal bar charts
    fig, axes = plt.subplots(n_sets, n_models + 1, figsize=figsize, constrained_layout=True)
    fig.suptitle(title, fontsize=20, fontweight='bold')

    all_evaluations = []

    for i, (model, m_name, s_dict) in enumerate(zip(model_list, model_names, all_sets_list)):
        # Step 1: Evaluate
        res = evaluate_model_on_sets(model, s_dict)
        all_evaluations.append(res)
        
        # Step 2: Plot Confusion Matrices in the first n_models columns
        for j, s_name in enumerate(set_names):
            ax = axes[j, i]
            sns.heatmap(res['cms'][s_name], annot=True, fmt='d', cmap='Blues', ax=ax, cbar=False)
            ax.set_title(f"{m_name} - {s_name}")
            ax.set_xlabel("Predicted")
            ax.set_ylabel("True")

    # Step 3: Plot Horizontal Stats in the last column
    for j, s_name in enumerate(set_names):
        ax = axes[j, -1]
        
        # Compile stats for this specific row (set) across all models
        stats_data = []
        for idx, m_name in enumerate(model_names):
            m_metrics = all_evaluations[idx]['metrics_df'].loc[s_name]
            for metric_name in ['F1', 'Accuracy']:
                stats_data.append({'Model': m_name, 'Metric': metric_name, 'Score': m_metrics[metric_name]})
        
        df_plot = pd.DataFrame(stats_data)
        sns.barplot(data=df_plot, x='Score', y='Metric', hue='Model', ax=ax)
        ax.set_xlim(0, 1.0)
        ax.set_title(f"Global Stats - {s_name}")
        ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', borderaxespad=0.)

    return fig

def separe_dataset(dataset, target='is_reference_valid'):
    """ 
    Divide features (X) and targets (y).
    """
    X = dataset.drop(target, axis=1, inplace=False)
    y = dataset[target]

    return X, y

def set_dict(datasets, names=['train', 'test'], target='is_reference_valid'):
    """
    Prepares a dictionary of (X, y) pairs for a model's evaluation.
    
    Args:
        datasets: List of DataFrames [df_train, df_test]
        names: Keys for the dictionary (e.g., ['train', 'test'])
    Returns:
        A dictionary: {'train': (X_train, y_train), 'test': (X_test, y_test)}
    """
    res_dict = {}
    for df, name in zip(datasets, names):
        X, y = separe_dataset(df, target=target)
        res_dict[name] = (X, y)
    
    return res_dict   


# Models Comparison


## 1. Import Pretrained Models

In [ ]:
# Usage
models_dict = load_latest_models(MODELS_SAVE_PATH)

initial_feat_knn = models_dict['initial_features']['KNN']
initial_feat_xgb = models_dict['initial_features']['XGB']
initial_feat_transformer = models_dict['initial_features']['transformer']

graph_feat_knn = models_dict['graph_features']['KNN']
graph_feat_xgb = models_dict['graph_features']['XGB']
graph_feat_transformer = models_dict['graph_features']['transformer']

text64_feat_knn = models_dict['textual_embeddings_64']['KNN']
text64_feat_xgb = models_dict['textual_embeddings_64']['XGB']
text64_feat_transformer = models_dict['textual_embeddings_64']['transformer']

text128_feat_knn = models_dict['textual_embeddings_128']['KNN']
text128_feat_xgb = models_dict['textual_embeddings_128']['XGB']
text128_feat_transformer = models_dict['textual_embeddings_128']['transformer']

all_feat_knn = models_dict['all_features']['KNN']
all_feat_xgb = models_dict['all_features']['XGB']
all_feat_transformer = models_dict['all_features']['transformer']

## 2. Comparison by feature types

### 2.1 Initial Features

In [ ]:
sets_for_normal = set_dict(
    datasets=[norm_train, norm_test], 
    names=['train', 'test']
)

# 2. Define the models and names
initial_feat_models = [initial_feat_knn, initial_feat_xgb, initial_feat_transformer]
name_tag = 'Normal'
initial_feat_names = [f'KNN ({name_tag})', f'XGB ({name_tag})', f'Transformer ({name_tag})']

# 3. Create the list of datasets (one entry per model in initial_feat_models)
# Since they all use the same data, we repeat the dictionary
initial_feat_data_sets = [sets_for_normal] * len(initial_feat_models)

# 4. Generate the Visualization
initial_feat_fig = plot_model_comparison(
    model_list=initial_feat_models, 
    model_names=initial_feat_names, 
    all_sets_list=initial_feat_data_sets, 
    title=f"{name_tag} Features - Evaluation Results", 
    figsize=(20, 16) 
)

plt.show()

if SAVE:
    initial_feat_fig.figsave(PATH_PLOTS / f'eval_models_{name_tag.low()}.png')

### 2.2 Graph Features

In [ ]:
sets_for_graph = set_dict(
    datasets=[graph_train, graph_test], 
    names=['train', 'test']
)

# 2. Define the models and names
graph_feat_models = [graph_feat_knn, graph_feat_xgb, graph_feat_transformer]
name_tag = 'Graph'
graph_feat_names = [f'KNN ({name_tag})', f'XGB ({name_tag})', f'Transformer ({name_tag})']

# 3. Create the list of datasets (one entry per model in graph_feat_models)
# Since they all use the same data, we repeat the dictionary
graph_feat_data_sets = [sets_for_graph] * len(graph_feat_models)

# 4. Generate the Visualization
graph_feat_fig = plot_model_comparison(
    model_list=graph_feat_models, 
    model_names=graph_feat_names, 
    all_sets_list=graph_feat_data_sets, 
    title=f"{name_tag} Features - Evaluation Results", 
    figsize=(20, 16) 
)

plt.show()

if SAVE:
    graph_feat_fig.figsave(PATH_PLOTS / f'eval_models_{name_tag.low()}.png')

### 2.3 Textual features

In [ ]:
sets_for_text64 = set_dict(
    datasets=[text64_train, text64_test], 
    names=['train', 'test']
)

# 2. Define the models and names
text64_feat_models = [text64_feat_knn, text64_feat_xgb, text64_feat_transformer]
name_tag = 'Textual 64'
text64_feat_names = [f'KNN ({name_tag})', f'XGB ({name_tag})', f'Transformer ({name_tag})']

# 3. Create the list of datasets (one entry per model in text64_feat_models)
# Since they all use the same data, we repeat the dictionary
text64_feat_data_sets = [sets_for_text64] * len(text64_feat_models)

# 4. Generate the Visualization
text64_feat_fig = plot_model_comparison(
    model_list=text64_feat_models, 
    model_names=text64_feat_names, 
    all_sets_list=text64_feat_data_sets, 
    title=f"{name_tag} Features - Evaluation Results", 
    figsize=(20, 16) 
)

plt.show()

if SAVE:
    text64_feat_fig.figsave(PATH_PLOTS / f'eval_models_{name_tag.low()}.png')

In [ ]:
sets_for_text128 = set_dict(
    datasets=[text128_train, text128_test], 
    names=['train', 'test']
)

# 2. Define the models and names
text128_feat_models = [text128_feat_knn, text128_feat_xgb, text128_feat_transformer]
name_tag = 'text128'
text128_feat_names = [f'KNN ({name_tag})', f'XGB ({name_tag})', f'Transformer ({name_tag})']

# 3. Create the list of datasets (one entry per model in text128_feat_models)
# Since they all use the same data, we repeat the dictionary
text128_feat_data_sets = [sets_for_text128] * len(text128_feat_models)

# 4. Generate the Visualization
text128_feat_fig = plot_model_comparison(
    model_list=text128_feat_models, 
    model_names=text128_feat_names, 
    all_sets_list=text128_feat_data_sets, 
    title=f"{name_tag} Features - Evaluation Results", 
    figsize=(20, 16) 
)

plt.show()

if SAVE:
    text128_feat_fig.figsave(PATH_PLOTS / f'eval_models_{name_tag.low()}.png')

### 2.4 Combined features

In [ ]:
sets_for_mix = set_dict(
    datasets=[mix_train, mix_test], 
    names=['train', 'test']
)

# 2. Define the models and names
mix_feat_models = [all_feat_knn, all_feat_xgb, all_feat_transformer]
name_tag = 'mix'
mix_feat_names = [f'KNN ({name_tag})', f'XGB ({name_tag})', f'Transformer ({name_tag})']

# 3. Create the list of datasets (one entry per model in mix_feat_models)
# Since they all use the same data, we repeat the dictionary
mix_feat_data_sets = [sets_for_mix] * len(mix_feat_models)

# 4. Generate the Visualization
mix_feat_fig = plot_model_comparison(
    model_list=mix_feat_models, 
    model_names=mix_feat_names, 
    all_sets_list=mix_feat_data_sets, 
    title=f"{name_tag} Features - Evaluation Results", 
    figsize=(20, 16) 
)

plt.show()

if SAVE:
    mix_feat_fig.figsave(PATH_PLOTS / f'eval_models_{name_tag.low()}.png')

## 3. Global Comparison

### 3.1 All figures

In [ ]:
display(initial_feat_fig, graph_feat_fig, text64_feat_fig, text128_feat_fig, mix_feat_fig)

> TODO comments

### 3.2 Heatmap of the stats

In [ ]:
#TODO: heatmap of the stats (each row is a model and each column a stat)

In [ ]:
def plot_global_stats_heatmap(model_groups, group_names, titles, figsize=(12, 10)):
    """
    Collects stats from all models and plots a global heatmap.
    """
    global_stats = []
    
    for models, names, sets_list in zip(model_groups, group_names, titles):
        for model, m_name, s_dict in zip(models, names, sets_list):
            # Use the existing evaluation function
            res = evaluate_model_on_sets(model, s_dict)
            
            # We focus on 'test' set results for the global comparison
            if 'test' in res['metrics_df'].index:
                stats = res['metrics_df'].loc['test'].to_dict()
                stats['Model_Type'] = m_name
                global_stats.append(stats)

    # Create the summary DataFrame
    df_stats = pd.DataFrame(global_stats).set_index('Model_Type')
    
    # Generate the heatmap
    plt.figure(figsize=figsize)
    sns.heatmap(df_stats, annot=True, cmap='YlGnBu', fmt='.4f', cbar_kws={'label': 'Score'})
    plt.title("Global Model Comparison - Test Set Metrics", fontsize=16, fontweight='bold')
    plt.ylabel("Model (Feature Set)")
    plt.tight_layout()
    
    if SAVE:
        plt.savefig(PATH_PLOTS / 'global_stats_heatmap.png')
    
    plt.show()

In [ ]:
all_models_groups = [
    initial_feat_models, 
    graph_feat_models, 
    text64_feat_models, 
    text128_feat_models, 
    mix_feat_models
]

all_names_groups = [
    initial_feat_names, 
    graph_feat_names, 
    text64_feat_names, 
    text128_feat_names, 
    mix_feat_names
]

all_sets_groups = [
    initial_feat_data_sets, 
    graph_feat_data_sets, 
    text64_feat_data_sets, 
    text128_feat_data_sets, 
    mix_feat_data_sets
]

# Plot the global comparison heatmap
plot_global_stats_heatmap(all_models_groups, all_names_groups, all_sets_groups)

> TODO comments